# Three numbers per sensor

**Temporal-stage notebook 3 in `notebooks/temporal/`.** `notebooks/temporal/README.md` says what the notebooks in this folder share. This notebook builds on the two before it: `01_last_cycle_fd001.ipynb` for what a temporal stage receives and returns, and `02_mean_window_fd001.ipynb` for why an average trails a sensor that is changing.

## What this notebook is for

The stage in this notebook, `window_summary`, hands each sensor's node three numbers made from its 30 readings: the **newest reading**, the **average**, and the **change**, how far the best straight line through the 30 readings rises from the first cycle to the last. Where the sensor is, where it has been, and which way it is going.

It is not in the paper; this project added it to answer a question the paper raises (assumption A-28). The newest reading drops 29 of 30 readings, and the paper's recurrent stage keeps them but needs sizes the paper never gives. This stage asks a narrower question with nothing to learn: does the direction a reading is moving in carry information the newest reading alone does not?

## What it does not do

- **It does not train anything.** The stage has nothing to learn, and the network's results are read from stored runs.
- **It does not say which of the three numbers the network uses.** No explanation method has been run on this stage.
- **It does not compare this stage with another.** `07_comparison_fd001.ipynb` does that.

## The notebook in one list

| # | Step | What goes in | What comes out |
|---|------|--------------|----------------|
| 1 | **Load the window** | `config/fd001.yaml`, seed 42 | the held-back engine's last window |
| 2 | **The change, by hand** | T50's 30 readings | the best line's rise across the window, term by term, checked against the stage |
| 3 | **The three numbers over a life** | every window of the engine | how the three move as the engine wears |
| 4 | **The gate and the cost** | the network built with this stage | why it is built only when named, and its learned numbers |
| 5 | **Read the stored results** | `reports/` | every stored five-seed run of this stage under this folder's settings |

## 1. Background

### The best straight line through 30 readings

The **best straight line** through a set of points is the one that makes the sum of the squared vertical distances from the points to the line as small as possible. Its rise per cycle, the **slope**, has a short formula once the cycles are numbered from the middle of the window. Number them `t = −14.5, −13.5, …, +13.5, +14.5`: position 1 becomes −14.5, position 30 becomes +14.5, and the thirty values of `t` add up to zero. Then

```
slope  =  Σ t × reading  /  Σ t²          Σ t² = 2247.5 for these thirty values of t
change =  slope × 29
```

- `t` is how many cycles a reading sits after the window's middle (negative for readings before it);
- `reading` is the sensor's rescaled value at that cycle;
- each sum `Σ` runs over all thirty cycles;
- `29` is the number of cycles from the first reading to the last.

**Why numbering from the middle helps.** Raising every reading by the same amount `a` adds `a × Σ t` to the top of the fraction, and `Σ t` is zero, so the slope does not change. The slope therefore measures only which way the readings go, never where they sit, which is the average's job.

**A tiny example.** Four readings 1, 4, 4, 3 at `t = −1.5, −0.5, +0.5, +1.5`:

```
Σ t × reading  =  (−1.5 × 1) + (−0.5 × 4) + (0.5 × 4) + (1.5 × 3)  =  −1.5 − 2 + 2 + 4.5  =  3
Σ t²           =  2.25 + 0.25 + 0.25 + 2.25                         =  5
slope          =  3 / 5  =  0.6 per cycle
change         =  0.6 × 3  =  1.8          (three cycles from the first reading to the last)
```

The line passes through the average, 3, at the middle, so it runs from 3 − 1.5 × 0.6 = 2.1 to 3 + 1.5 × 0.6 = 3.9. It passes through neither the first reading nor the last, and its change, 1.8, is not the last reading minus the first, 2.

### Why the change and not the slope

All three numbers enter one weight matrix whose starting values are drawn on the same scale for every input (assumption A-14), and the graph layers carry no offset that could make up for a difference in size (assumption A-12). Measured per cycle, the slope is 29 times smaller than the change; multiplied by 29 it is in the readings' own units, like the other two numbers. Step 2 prints both for T50. That the scale matters is an argument, not a measurement: the per-cycle version was never trained (assumption A-28).

**Do not confuse** the change with the last reading minus the first. The first and last readings are two of thirty, and either can be unusual; the line uses all thirty.

## 2. Setup

In [ ]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

# The project's top folder is the nearest one above this notebook that holds
# pyproject.toml. Adding its src/ folder to Python's search path makes the
# package importable from a clone that was never installed.
PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "pyproject.toml").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    import torch
except ImportError as missing:
    raise ImportError(
        "This notebook builds the network's temporal stage, which needs PyTorch, and PyTorch is "
        'not installed. Install the project with  pip install -e ".[model]"'
    ) from missing

from thesis_gnn_rul_xai.config import load_config
from thesis_gnn_rul_xai.data.pipeline import run_preprocessing
from thesis_gnn_rul_xai.models import ModelConfig, build_model
from thesis_gnn_rul_xai.models import ESTGNN
from thesis_gnn_rul_xai.models.temporal import WindowSummary
from thesis_gnn_rul_xai.training import TrainingConfig, find_runs
from thesis_gnn_rul_xai.utils.logging_setup import configure_logging

configure_logging(logging.ERROR)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", None)    # a run's folder name is printed whole
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch     : {torch.__version__}")

## 3. The window

### Step 1 — Load the window

**What.** Rebuild the data from `config/fd001.yaml`, take the held-back engine with the most windows, and take its last window. Print the 30 readings of T50, the temperature at the outlet of the low-pressure turbine, the sensor every notebook in this folder follows.

**Why.** Every notebook here works on this same window, so a difference between two notebooks is a difference between two stages. `notebooks/temporal/README.md` says why this engine, this window and this sensor, and what the units of a reading are.

**Check.** The window must be 14 sensors by 30 cycles, come from an engine the training never saw, and end at the cycle the engine failed.

In [ ]:
config = load_config(PROJECT_ROOT / "config" / "fd001.yaml")
prepared = run_preprocessing(config)

sensors = list(prepared.sensor_symbols)

# The held-back engine with the most windows, and its last window.
held_back = prepared.validation
units, counts = np.unique(held_back.units, return_counts=True)
ENGINE = int(units[np.argmax(counts)])
engine_rows = np.flatnonzero(held_back.units == ENGINE)
engine_rows = engine_rows[np.argsort(held_back.end_cycles[engine_rows])]

SENSOR = "T50"
s = sensors.index(SENSOR)
window = held_back.features[engine_rows[-1]]             # (14 sensors, 30 cycles)
end_cycle = int(held_back.end_cycles[engine_rows[-1]])
cycles = np.arange(end_cycle - window.shape[1] + 1, end_cycle + 1)
readings = window[s]                                     # T50's 30 readings, oldest first

print(f"Engine {ENGINE}: {len(engine_rows)} windows, held back from training under seed {config.seed}.")
print(f"Worked window: cycles {cycles[0]} to {cycles[-1]}, shape {window.shape}, "
      f"remaining life at its end {held_back.targets[engine_rows[-1]]:.0f} cycles.\n")
display(pd.DataFrame([readings], index=[f"{SENSOR} reading"],
                     columns=pd.MultiIndex.from_arrays([range(1, 31), cycles],
                                                       names=["position", "engine cycle"])).round(6))

window_checks = {
    "the window is 14 sensors by 30 cycles": window.shape == (14, 30),
    "its engine is not a training engine": ENGINE not in set(prepared.train.units),
    "it ends at the cycle the engine failed": float(held_back.targets[engine_rows[-1]]) == 0.0,
}
assert all(window_checks.values()), window_checks
print("OK  " + "; ".join(window_checks))

## 4. The stage, step by step

### Step 2 — The change, by hand

**What.** Write out all thirty terms of `Σ t × reading` for T50, add them, divide by 2247.5, multiply by 29. The newest reading and the average are the two numbers notebooks 01 and 02 computed; the cell prints them again beside the change.

**Why.** The change is the one number of the three a reader cannot see at a glance, and each of its thirty terms shows how much one reading pulls the line up or down: a reading after the middle that sits high pulls it up, one before the middle that sits high pulls it down.

**Check.** The hand result must match `WindowSummary(30)`'s third number for T50, and NumPy's own line fit; `Σ t²` must be 2247.5.

In [ ]:
t = np.arange(30) - 14.5
terms = pd.DataFrame({"engine cycle": cycles, "t": t, f"{SENSOR} reading": readings,
                      "t × reading": t * readings.astype(np.float64)},
                     index=pd.Index(range(1, 31), name="position"))
display(terms.T.round(4))

sum_t_reading = float(terms["t × reading"].sum())
sum_t_squared = float((t ** 2).sum())
slope = sum_t_reading / sum_t_squared
change = slope * 29
print(f"Σ t × reading = {sum_t_reading:.6f}")
print(f"Σ t²          = {sum_t_squared}")
print(f"slope         = {sum_t_reading:.6f} / {sum_t_squared} = {slope:.6f} per cycle")
print(f"change        = {slope:.6f} × 29 = {change:.6f}\n")

stage = WindowSummary(window_size=30)
with torch.no_grad():
    handed_on = stage(torch.from_numpy(window[None]))      # (1, 14, 3)
print(f"in  {tuple(window[None].shape)}    out {tuple(handed_on.shape)}\n")
display(pd.DataFrame(handed_on[0].numpy(), index=sensors,
                     columns=["newest reading", "average", "change"]).T.round(6))

sum_is_2247_5 = sum_t_squared == 2247.5
hand_matches_stage = bool(abs(change - float(handed_on[0, s, 2])) < 1e-5)
numpy_slope = np.polyfit(np.arange(30), readings.astype(np.float64), deg=1)[0]
hand_matches_numpy = bool(abs(slope - numpy_slope) < 1e-9)
first_two_are_known = bool(np.allclose(handed_on[0, :, 0].numpy(), window[:, -1])
                           and np.allclose(handed_on[0, :, 1].numpy(), window.mean(axis=1), atol=1e-6))
assert sum_is_2247_5 and hand_matches_stage and hand_matches_numpy and first_two_are_known
print(f"OK  the stage's change for {SENSOR} is {float(handed_on[0, s, 2]):.6f}, as by hand; NumPy's "
      f"line rises {numpy_slope:.6f} per cycle; the first two numbers are the newest reading and the average.")

**Reading the terms.** The terms at the two ends of the window are the largest, because there `t` is largest in size. The four readings of cycles 266 to 269, all above 2.48, add between 28.5 and 39.7 each; the four of cycles 240 to 243, between 1.25 and 1.83, add between −14.4 and −24.7. A high reading late in the window and a low reading early both pull the line up. The thirty add up to 82.146504, which divided by 2247.5 is a rise of 0.036550 per cycle, or 1.059955 across the window.

The change, 1.060, is not the last reading minus the first, 2.740 − 1.292 = 1.448. Those two are two readings of thirty, while the line weighs every reading by how far it sits from the middle.

### Step 3 — The three numbers over an engine's life

**What.** Run the stage on every window of the engine and draw its three numbers for T50 against the cycle at which each window ends.

**Why.** One window shows the arithmetic. The life shows what the third number adds: whether the change says something the newest reading and the average do not.

In [ ]:
with torch.no_grad():
    life = stage(torch.from_numpy(held_back.features[engine_rows]))[:, s, :].numpy()   # (240, 3)
ends = held_back.end_cycles[engine_rows]
thirds = np.array_split(np.arange(len(engine_rows)), 3)
summary_by_third = pd.DataFrame(
    [life[rows].mean(axis=0) for rows in thirds],
    index=[f"windows ending at cycles {ends[rows[0]]} to {ends[rows[-1]]}" for rows in thirds],
    columns=["newest reading", "average", "change"])
print(f"T50's three numbers, averaged over each third of engine {ENGINE}'s {len(ends)} windows:")
display(summary_by_third.round(3))
print(f"The change is above zero in {(life[:, 2] > 0).sum()} of {len(ends)} windows.")

figure, axis = plt.subplots(figsize=(8, 3.4))
for column, name, colour in ((0, "newest reading", "C3"), (1, "average", "C0"), (2, "change", "C2")):
    axis.plot(ends, life[:, column], color=colour, linewidth=1.2, label=name)
axis.axhline(0, color="0.7", linewidth=0.8)
axis.set(title=f"WindowSummary on every window of engine {ENGINE}, sensor {SENSOR}",
         xlabel="cycle at which the window ends", ylabel="rescaled reading units")
axis.legend()
figure.tight_layout()
plt.show()

Averaged over each third of the engine's windows, T50's change is 0.165, 0.241 and 0.836: positive in each third, and largest in the last, where T50 climbs fastest. It also moves a good deal within a third — a line through 30 readings that scatter is itself unsteady — and it is above zero in 194 of the 240 windows.

The newest reading says where T50 is and the average where it has been; the change adds how fast it was moving, which the newest reading cannot show and the average shows only as a lag (`02_mean_window_fd001.ipynb`, step 3).

### Step 4 — The gate, and what it costs

**What.** First try to build the network with this stage the way a caller might by accident, handing `ESTGNN` a `WindowSummary` directly; the network refuses. Then build it the way `scripts/train.py` does, through `build_model` with the stage named, and count its learned numbers.

**Why.** With three numbers per sensor the first graph layer's weight matrix becomes `(3, 32)`. Its rows multiply numbers made from the cycles, so that layer now mixes information across time without learning anything before it — the same thing that went unnoticed until 2026-08-28, when it was found that 30 raw cycles had been reaching the first graph layer (assumption A-07). The network therefore asks whether its stage holds learned numbers; a stage that hands on more than one number and learns nothing is built only when named, and a run that names it records the name. This stage is one of those.

**Check.** The direct build must be refused; the named build must give `W(1)` of shape `(3, 32)`, no learned numbers in the stage, and 61,729 + 2 × 32 = 61,793 in the whole network.

In [ ]:
TABLE_4_NAMES = {"graph_layers.0": "GCN-1", "graph_layers.1": "GCN-2",
                 "graph_layers.2": "GCN-3", "fc1": "FC-1", "fc2": "FC-2"}


def parameter_table(network):
    """Learned numbers in each part of a network, the temporal stage first."""
    parts = {"temporal stage": 0}
    for name, parameter in network.named_parameters():
        part = "temporal stage" if name.startswith("temporal.") else TABLE_4_NAMES[name.rsplit(".", 1)[0]]
        parts[part] = parts.get(part, 0) + parameter.numel()
    parts["whole network"] = sum(parts.values())
    return pd.DataFrame({"learned numbers": parts})


try:
    ESTGNN(prepared.adjacency_normalized, temporal=WindowSummary(30), rul_max=config.rul_max)
    refused = False
except ValueError as refusal:
    refused = True
    print("Refused:", str(refusal).split(". ")[0] + ".\n")

network = build_model(prepared.adjacency_normalized, ModelConfig(temporal="window_summary"),
                      window_size=config.window_size, rul_max=config.rul_max)
costs = parameter_table(network)
display(costs.T)
cost_checks = {
    "the direct build is refused": refused,
    "the stage learns nothing": int(costs.loc["temporal stage", "learned numbers"]) == 0,
    "W(1) is (3, 32)": tuple(network.graph_layers[0].weight.shape) == (3, 32),
    "the whole network holds 61,793": int(costs.loc["whole network", "learned numbers"]) == 61_729 + 2 * 32,
}
assert all(cost_checks.values()), cost_checks
print("OK  " + "; ".join(cost_checks))

### Step 5 — The stored results of this stage

**What.** Every stored five-seed run that used this stage under this folder's settings. Runs that saved only part of their settings — the oldest saved nothing about their training but the learning-rate path — are listed with what they did not save, and their seeds are set beside those of the runs that saved everything.

**Why.** As for every stage, an error in cycles belongs to the whole trained network and its settings.

In [ ]:
STUDY = PROJECT_ROOT / "reports" / "sensitivity_fd65a3b-dirty_20260929T101059Z" / "runs" / "FD001"
DEFAULT_TRAINING = TrainingConfig()
TRAINING_FIELDS = ("epochs", "batch_size", "learning_rate", "final_learning_rate",
                   "early_stopping_patience", "lr_schedule")
WANTED = [ModelConfig(temporal="window_summary")]


def training_settings(run) -> str:
    """'default' or 'other' from the training settings a run saved; 'not saved' if it saved only the learning-rate path."""
    recorded = run.summary.get("training")
    if recorded is None:
        return "not saved" if run.summary.get("lr_schedule") == DEFAULT_TRAINING.lr_schedule else "other"
    default = all(recorded.get(field) == getattr(DEFAULT_TRAINING, field) for field in TRAINING_FIELDS)
    return "default" if default else "other"


def not_saved(run) -> str:
    """The settings a run did not save, or a dash if it saved them all."""
    missing = [] if run.summary.get("training") else ["training settings"]
    if not (run.path / "model_config.yaml").is_file():
        missing.append("network sizes" if "temporal_stage" in run.summary else "network sizes, stage")
    return ", ".join(missing) or "—"


def machine(run) -> str:
    """The device a run's first seed asked for, and the PyTorch build the run recorded."""
    saved = run.path / f"seed_{run.seeds[0]}" / "training_config.yaml"
    device = yaml.safe_load(saved.read_text(encoding="utf-8")).get("device") if saved.is_file() else None
    build = ((run.summary.get("environment") or {}).get("packages") or {}).get("torch")
    return f"{device or 'not recorded'}, PyTorch {build or 'not recorded'}"


stored = [
    run
    for root in (None, STUDY)
    for run in find_runs(root=root, preprocessing=config, seeds=5, split="varies_with_seed",
                         early_stopping=True)
    if run.model_config in WANTED and training_settings(run) != "other"
]
print("Settings every run below shares, where it saved them:")
print("  preprocessing:", {k: v for k, v in config.to_dict().items() if k != "notes"})
print("  training:     ", {field: getattr(DEFAULT_TRAINING, field) for field in TRAINING_FIELDS},
      "and the pass that scored best on the held-back engines restored")
for model_config in WANTED:
    print("  network:      ", model_config.to_dict())
print()
if stored:
    display(pd.DataFrame([{
        "run, under reports/": run.path.resolve().relative_to((PROJECT_ROOT / "reports").resolve()).as_posix(),
        "commit": run.summary["commit"],
        **({"one network for all sensors": run.model_config.shared_across_sensors}
           if len(WANTED) > 1 else {}),
        "device asked for, PyTorch build": machine(run),
        "not saved by the run": not_saved(run),
        "seeds": ", ".join(map(str, run.seeds)),
        "test RMSE": f"{run.results['rmse'].mean():.3f} ± {run.results['rmse'].std():.3f}",
        "held-back RMSE": f"{run.results['validation_rmse'].mean():.3f} ± "
                          f"{run.results['validation_rmse'].std():.3f}",
    } for run in stored]).set_index("run, under reports/"))
else:
    print("reports/ holds no five-seed run of this stage under these settings. Make one with\n\n"
          "    python scripts/train.py --config config/fd001.yaml --temporal window_summary --seeds 5\n")

# A run that saved less is set beside the runs that saved everything, seed by seed.
complete = [run for run in stored if not_saved(run) == "—"]
repeats = {}
for run in stored:
    if not_saved(run) == "—":
        continue
    mine = run.results.set_index("seed").sort_index()[["rmse", "validation_rmse"]].to_numpy()
    repeats[run.name] = [
        other.summary["commit"] for other in complete
        if np.array_equal(other.results.set_index("seed").sort_index()[["rmse", "validation_rmse"]].to_numpy(), mine)
    ]
    unsaid = not_saved(run).split(", ")
    unsaid = unsaid[0] if len(unsaid) == 1 else ", ".join(unsaid[:-1]) + " or " + unsaid[-1]
    twins = repeats[run.name]
    print(f"The run at {run.summary['commit']} saved no {unsaid}; its five seeds give the same test and held-back "
          f"RMSE, to every stored digit, as the run{'s' if len(twins) > 1 else ''} at {', '.join(twins) or 'no commit'}.")
unsaved_runs_repeat = all(repeats.values())

**One stored run**, at `5602773-dirty`, under the settings printed above. A run of the newest-reading stage was made from the same tree at the same commit, so the two can be paired seed by seed; `07_comparison_fd001.ipynb` does that.

**Which published row it may be set beside: none.** The paper describes no stage like this one, so none of its rows was produced by anything like it (assumption A-28).

## Final verification

In [ ]:
checks = [
    ("The window is the folder's window", all(window_checks.values()), f"engine {ENGINE}, cycle {end_cycle}"),
    ("Σ t² over thirty centred cycles is 2247.5", sum_is_2247_5, f"{sum_t_squared}"),
    ("The change by hand matches the stage", hand_matches_stage, f"{change:.6f}"),
    ("The slope by hand matches NumPy's line", hand_matches_numpy, f"{slope:.6f} per cycle"),
    ("The first two numbers are newest and average", first_two_are_known, "14 sensors"),
    ("The gate and the cost", all(cost_checks.values()), "refused unnamed; 61,793 learned numbers"),
    ("Each run that saved less repeats one that saved all", unsaved_runs_repeat, f"{len(repeats)} such run(s)"),
]
width = max(len(name) for name, _, _ in checks) + 2
print(f"{'CHECK':<{width}} RESULT   DETAIL")
print("-" * (width + 40))
for name, passed, detail in checks:
    print(f"{name:<{width}} {'PASS' if passed else 'FAIL':<8} {detail}")
print("-" * (width + 40))
failed = [name for name, passed, _ in checks if not passed]
if failed:
    raise AssertionError(f"{len(failed)} check(s) failed: {failed}")
print(f"All {len(checks)} checks passed.")
print(f"\nStored five-seed runs of this stage under this folder's settings: {len(stored)}; the step on stored results lists them, or prints the command that makes one.")

## Summary

### What this notebook produced

- **The change** for T50 on the worked window, its thirty terms written out and checked against the stage and against NumPy.
- **The three numbers over an engine's life**, showing when the change departs from zero.
- **The gate**: this stage is built only when named, because it hands the first graph layer three numbers made from the cycles while learning nothing itself.
- **Its cost**: 64 learned numbers more than Table 4's network, all in the first graph layer.
- **Its stored results**, with their commit and settings.

### Decisions that were not read from the paper

| Decision | Value used | Written up as |
|---|---|---|
| Giving each sensor three numbers | newest reading, average, change across the window | A-28 |
| The change in reading units rather than per cycle | slope × 29 | A-28 |
| Which window and which sensor are shown | the held-back engine's last window, T50 | `notebooks/temporal/README.md` |

### References

- Wen, J.-R., et al. (2026). Explainable spatiotemporal graph neutral network with dynamic SHAP enhancement for aeroengine RUL prediction. *Aerospace Science and Technology*, 168, 110875. Table 4.